# Download and import the required libraries

In [ ]:
import os
import torch
os.environ['TORCH'] = torch.__version__

!pip install torch_geometric
!pip install -q torch-scatter -f https://data.pyg.org/whl/torch-${TORCH}.html
!pip install -q torch-sparse -f https://data.pyg.org/whl/torch-${TORCH}.html
!pip install -q git+https://github.com/pyg-team/pytorch_geometric.git

import numpy as np
import time
import warnings
import torch.nn.functional as F
from torch.nn import Linear, Sequential, BatchNorm1d, ReLU
from torch_geometric.loader import DataLoader
from torch_geometric.datasets import TUDataset
from torch_geometric.transforms import OneHotDegree
from torch_geometric.utils import degree
from torch_geometric.nn import GINConv, global_add_pool, global_mean_pool
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score
from sklearn.model_selection import StratifiedKFold
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, SpectralClustering
from sklearn.metrics import adjusted_rand_score
from sklearn.manifold import TSNE
try:
    import umap.umap_ as umap
except ImportError:
    import umap
import random
import copy

os.makedirs('./best_models', exist_ok=True)
os.makedirs('./plots', exist_ok=True)

# Load the Dataset

In [ ]:
import torch_geometric.transforms as T

def get_dataset(name):
    if (name == 'MUTAG'):
        dataset = TUDataset(root='./data', name=name, use_node_attr=True)
        
    elif (name=='ENZYMES'):
        dataset = TUDataset(
            root='./data', 
            name='ENZYMES',
            use_node_attr=True,  # Ensure continuous features are loaded
            transform=T.Compose([
                T.NormalizeFeatures(), # Normalize the continuous features
                T.OneHotDegree(max_degree=15) # Adds structural degree info
            ])
        )

    else: 
        dataset = TUDataset(root='./data', name=name, use_node_attr=True)

        # Handle datasets with no node feature by adding one-hot encoded node degrees as features
        max_degree = 0
        for data in dataset:
            d = degree(data.edge_index[0], dtype=torch.long)
            max_degree = max(max_degree, int(d.max()))
        
        # Apply the transform to generate features
        dataset.transform = OneHotDegree(max_degree)
    
    return dataset

# GIN Architecture

In [ ]:
class GIN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, dropout=0.5, num_layers=3):
        super().__init__()
        self.dropout = dropout
        self.num_layers = num_layers

        # Layer 0: Mapping input features to hidden dimension
        # so it has the same weight as the rest of the layers in the final embedding
        self.conv_zero = Sequential(
            Linear(in_channels, hidden_channels),
            BatchNorm1d(hidden_channels),
            ReLU()
        )

        # Layers 1 - K: GIN convolutions
        self.convs = torch.nn.ModuleList()
        for _ in range(num_layers):
            mlp = Sequential(
                Linear(hidden_channels, hidden_channels),
                BatchNorm1d(hidden_channels),
                ReLU(),
                Linear(hidden_channels, hidden_channels),
                BatchNorm1d(hidden_channels),
                ReLU()
            )
            conv = GINConv(mlp, train_eps=True)
            self.convs.append(conv)
        
        # Classifier
        # Input dim is hidden_channels*(num_layers+1) because we concat Layer 0 + K layers
        self.lin1 = Linear(hidden_channels*(num_layers+1), hidden_channels)
        self.lin2 = Linear(hidden_channels, out_channels)
    
    def forward(self, x, edge_index, batch):
        x = self.conv_zero(x)
        node_embeddings = [x]

        # Node Embeddings (for every GIN Layer)
        # [nodes_in_batch, hidden_dimension] * (num_layers + 1)
        for conv in self.convs:
            x = conv(x, edge_index)
            x = F.dropout(x, p=self.dropout, training=self.training)
            node_embeddings.append(x)

        # Graph-level readout of node embeddings to produce Graph Embeddings (at each layer)
        # We avg the node embeddings at each graph to produce a Graph Embedding 
        # [batch_size, hidden_dimension] * (num_layers + 1)
        pooled = [global_mean_pool(h, batch) for h in node_embeddings]
            
        # Concatenate graph embeddings from all layers
        # [batch_size, hidden_dimension * (num_layers + 1)] = [batch_size, embedding_size]
        graph_embedding = torch.cat(pooled, dim=1)

        # Classifier
        h = self.lin1(graph_embedding)
        h = h.relu()
        h = F.dropout(h, p=self.dropout, training=self.training)
        out = self.lin2(h)

        return F.log_softmax(out, dim=1), F.softmax(out, dim=1), graph_embedding


def train(model, train_loader, optimizer):
    model.train()
    total_loss = 0
    total_graphs = 0
    
    for data in train_loader:
        data = data.to(device)
        optimizer.zero_grad()
        
        out, _, _ = model(data.x, data.edge_index, data.batch)
        
        loss = F.nll_loss(out, data.y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
        optimizer.step()
        
        total_loss += loss.item() * data.num_graphs 
        total_graphs += data.num_graphs
        
    return total_loss / total_graphs

def test(model, loader):
    model.eval()
    prediction_list, label_list, probabilities_list, embedding_list = [], [], [], []

    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            _, probabilities, embeddings = model(data.x, data.edge_index, data.batch)
            predictions = probabilities.argmax(dim=1)
            
            prediction_list.append(predictions.detach().cpu())
            label_list.append(data.y.detach().cpu())
            probabilities_list.append(probabilities.detach().cpu())
            embedding_list.append(embeddings.detach().cpu())
        
    y_pred = torch.cat(prediction_list).numpy()
    y_true = torch.cat(label_list).numpy()
    y_probs = torch.cat(probabilities_list).numpy()
    final_embeddings = torch.cat(embedding_list)
    
    return y_true, y_pred, y_probs, final_embeddings

# Classification

In [ ]:
# CONFIGURATION
DATASET_NAME = 'MUTAG'  # Options: 'MUTAG', 'ENZYMES', 'IMDB-MULTI'
BATCH_SIZE = 64
EMBEDDING_SIZE = 256
# Our GIN implementation has 4 components (Layer0 + 3 GIN Layers) and the final embedding
# is the concatenation of each component's embedding, so the final embedding size is hidden_dim * 4
HIDDEN_CHANNELS = int(EMBEDDING_SIZE/4) 
DROPOUT = 0.5
EPOCHS = 150
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
!mkdir best_models 

# LOAD DATA
dataset = get_dataset(DATASET_NAME)
X = np.zeros(len(dataset)) # Dummy features for splitter
y = np.array([data.y.item() for data in dataset]) # Labels for stratification
print(f"Dataset: {DATASET_NAME}")
print(f"Number of graphs: {len(dataset)}")
print(f"Number of classes: {dataset.num_classes}")
print(f"Number of node features: {dataset.num_features}")
print(f"Device: {device}")
print("-" * 60)

# SETUP 10-FOLD SPLITTER
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Lists to store results across 10 folds
results = {
    'acc': [], 'f1': [], 'auc': [], 
    'train_time': [], 'infer_time': [], 'memory': []
}
top_acc_so_far = 0

# CROSS VALIDATION LOOP
for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
    # SPLIT DATA
    train_dataset = dataset[torch.tensor(train_idx)]
    test_dataset = dataset[torch.tensor(test_idx)]
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

    # SETUP MODEL
    model = GIN(
        in_channels=dataset.num_features, 
        hidden_channels=HIDDEN_CHANNELS, 
        out_channels=dataset.num_classes, 
        dropout=DROPOUT,
        num_layers=3,
    ).to(device)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, min_lr=0.00001)

    # TRAINING LOOP
    best_fold_metrics = {'acc': 0.0, 'f1': 0.0, 'auc': 0.0, 'infer_time': 0.0}
    total_train_time = 0.0
    
    # Reset GPU memory tracking for this fold
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    for epoch in range(EPOCHS): 
        # Measure Training Time
        start_train = time.time()
        train_loss = train(model, train_loader, optimizer)
        total_train_time += (time.time() - start_train)
        
        # Measure Generation (Inference) Time
        start_infer = time.time()
        y_true, y_pred, y_probs, _ = test(model, test_loader)
        infer_time = time.time() - start_infer
        
        test_acc = (y_pred == y_true).sum() / len(y_true)
        scheduler.step(train_loss)     

        # If this is the best model so far (in this fold), calculate and store all detailed metrics
        if test_acc > best_fold_metrics['acc']:
            best_fold_metrics['acc'] = test_acc
            best_fold_metrics['infer_time'] = infer_time
            
            # Calculate F1 and AUC
            try:
                if dataset.num_classes == 2:
                    # For binary, pass only the positive class probability
                    best_fold_metrics['auc'] = roc_auc_score(y_true, y_probs[:, 1])
                    best_fold_metrics['f1'] = f1_score(y_true, y_pred, average='binary')
                else:
                    best_fold_metrics['auc'] = roc_auc_score(y_true, y_probs, multi_class='ovr', average='weighted')
                    best_fold_metrics['f1'] = f1_score(y_true, y_pred, average='weighted')
            except ValueError:
                best_fold_metrics['auc'] = 0.5

            # Save the best model across folds
            if test_acc > top_acc_so_far:
                top_acc_so_far = test_acc
                torch.save(model.state_dict(), './best_models/best_model.pth')
                
    # Record Memory (Peak VRAM used during this fold in MB)
    max_memory = torch.cuda.max_memory_allocated() / 1024**2 if torch.cuda.is_available() else 0

    # Append Fold Results
    results['acc'].append(best_fold_metrics['acc'])
    results['f1'].append(best_fold_metrics['f1'])
    results['auc'].append(best_fold_metrics['auc'])
    results['train_time'].append(total_train_time)
    results['infer_time'].append(best_fold_metrics['infer_time'])
    results['memory'].append(max_memory)
    
    print(f"[Fold {fold+1:02d}] Acc: {best_fold_metrics['acc']:.4f} | F1: {best_fold_metrics['f1']:.4f} | AUC: {best_fold_metrics['auc']:.2f} | Time: {total_train_time:.2f}s")

# PRINT FINAL RESULTS
print("-" * 60)
print(f"Final 10-Fold Results ({DATASET_NAME} - ΕΜΒ {EMBEDDING_SIZE} - {EPOCHS} EPOCHS):")
print(f"Accuracy:        {np.mean(results['acc'])*100:.2f}% ± {np.std(results['acc'])*100:.2f}")
print(f"F1-Score:        {np.mean(results['f1']):.4f} ± {np.std(results['f1']):.4f}")
print(f"AUC:             {np.mean(results['auc']):.4f} ± {np.std(results['auc']):.4f}")
print(f"Total Train Time:{np.mean(results['train_time']):.2f}s ± {np.std(results['train_time']):.2f}")
print(f"Generation Time: {np.mean(results['infer_time']):.4f}s ± {np.std(results['infer_time']):.4f}")
print(f"Peak Memory Use: {np.mean(results['memory']):.2f} MB")
print("-" * 60)

# Save the average baseline accuracy for comparison in the stability analysis
avg_baseline_acc = np.mean(results['acc'])

# Clustering 

In [ ]:
# EXTRACT GRAPH EMBEDDINGS FROM THE ENTIRE DATASETS
full_loader = DataLoader(dataset, batch_size=64, shuffle=False) # no shuffle for consistent plotting

# Get Embeddings using the best model from training
model.load_state_dict(torch.load('./best_models/best_model.pth'))
all_embeddings = []
all_labels = []

with torch.no_grad():
    for data in full_loader:
        data = data.to(device)
        _, _, embeddings = model(data.x, data.edge_index, data.batch)
        all_embeddings.append(embeddings.cpu().numpy())
        all_labels.append(data.y.cpu().numpy())

X_emb = np.concatenate(all_embeddings, axis=0)
y_true = np.concatenate(all_labels, axis=0)

print(f"Graph Embeddings Shape: {X_emb.shape}")

# CLUSTERING & METRICS (ARI)
num_classes = dataset.num_classes

# K-Means
kmeans = KMeans(n_clusters=num_classes, random_state=42, n_init=20)
y_kmeans = kmeans.fit_predict(X_emb)
ari_kmeans = adjusted_rand_score(y_true, y_kmeans)

# Spectral Clustering
spectral = SpectralClustering(n_clusters=num_classes, assign_labels='cluster_qr', random_state=42)
y_spectral = spectral.fit_predict(X_emb)
ari_spectral = adjusted_rand_score(y_true, y_spectral)

print("-" * 29)
print(f"Clustering Performance (ARI):")
print(f"K-Means ARI:              {ari_kmeans:.4f}")
print(f"Spectral Clustering ARI:  {ari_spectral:.4f}")
print("-" * 29)

# VISUALIZATION (t-SNE & UMAP)
tsne = TSNE(n_components=2, random_state=42)
z_tsne = tsne.fit_transform(X_emb)

umap_model = umap.UMAP(n_components=2, random_state=42)
z_umap = umap_model.fit_transform(X_emb)

def plot_embedding(z, labels, title, filename):
    plt.figure(figsize=(10, 8))
    sns.scatterplot(x=z[:,0], y=z[:,1], hue=labels, palette='tab10', s=60, alpha=0.8)
    plt.title(title, fontsize=16)
    plt.legend(title='Classes', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig(f"./plots/{filename}")
    plt.show()
    print(f"Saved: {filename}")

warnings.filterwarnings('ignore', message='.*n_jobs value 1 overridden.*')
plot_embedding(z_tsne, y_true, "t-SNE Visualization", f"tSNE_{DATASET_NAME}_{EMBEDDING_SIZE}")
plot_embedding(z_umap, y_true, "UMAP Visualization", f"UMAP_{DATASET_NAME}_{EMBEDDING_SIZE}")

# QUALITATIVE ASSESSMENT
def assess_separation(ari_score):
    if ari_score > 0.5:
        return "Clear Separation"
    elif ari_score > 0.2:
        return "Medium Separation"
    else:
        return "No/Poor Separation"

quality = assess_separation(max(ari_kmeans, ari_spectral))
print("-" * 41)
print(f"QUALITATIVE ASSESSMENT: {quality}")
print("-" * 41)

# Stability Analysis

In [ ]:
# PERTURBATION FUNCTIONS
def perturb_add_edges(data, noise_level):
    """Adds noise_level * num_edges random edges."""
    if noise_level == 0: return data
    data = data.clone()
    num_nodes = data.num_nodes
    num_existing = data.edge_index.size(1)
    num_to_add = int(num_existing * noise_level)
    
    if num_to_add > 0:
        row = torch.randint(0, num_nodes, (num_to_add,), dtype=torch.long)
        col = torch.randint(0, num_nodes, (num_to_add,), dtype=torch.long)
        mask = row != col # Avoid self-loops
        new_edges = torch.stack([row[mask], col[mask]], dim=0)
        data.edge_index = torch.cat([data.edge_index, new_edges], dim=1)
    return data

def perturb_drop_edges(data, noise_level):
    """Removes noise_level * num_edges existing edges."""
    if noise_level == 0: return data
    data = data.clone()
    num_edges = data.edge_index.size(1)
    num_to_drop = int(num_edges * noise_level)
    
    if num_to_drop > 0:
        # We prefer to keep edges, so we calculate how many to keep
        num_keep = num_edges - num_to_drop
        # Random permutation of indices
        perm = torch.randperm(num_edges)
        keep_indices = perm[:num_keep]
        data.edge_index = data.edge_index[:, keep_indices]
    return data

def perturb_add_drop_edges(data, noise_level):
    """
    Simultaneously adds (noise/2) edges and drops (noise/2) edges.
    Example: noise_level 0.05 adds 2.5% edges and drops 2.5% edges.
    """
    if noise_level == 0: return data
    
    # Split the budget
    half_noise = noise_level / 2.0
    
    # 1. Add Edges first
    data = perturb_add_edges(data, half_noise)
    
    # 2. Drop Edges (from the new graph)
    data = perturb_drop_edges(data, half_noise)
    
    return data

def perturb_shuffle_features(data, noise_level):
    """Shuffles the feature vectors of noise_level * num_nodes nodes."""
    if noise_level == 0: return data
    # If dataset has no features (like IMDB-MULTI raw), skip or handle
    if data.x is None: return data
    
    data = data.clone()
    num_nodes = data.num_nodes
    num_to_shuffle = int(num_nodes * noise_level)
    
    if num_to_shuffle > 1:
        # Select random nodes to shuffle
        perm = torch.randperm(num_nodes)
        nodes_to_shuffle = perm[:num_to_shuffle]
        
        # Get their features
        features = data.x[nodes_to_shuffle]
        
        # Shuffle these features among themselves
        shuffled_indices = torch.randperm(num_to_shuffle)
        data.x[nodes_to_shuffle] = features[shuffled_indices]
        
    return data

# Dictionary mapping perturbation names to functions
perturbation_types = {
    'Add Edges': perturb_add_edges,
    'Drop Edges': perturb_drop_edges,
    'Mixed (Add+Drop)': perturb_add_drop_edges,  
    'Shuffle Features': perturb_shuffle_features
}


# RUN EXPERIMENTS
noise_levels = [0.00, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
results = {name: {'stability': [], 'accuracy': []} for name in perturbation_types}

# Loop Over Perturbations
for p_name, p_func in perturbation_types.items():
    print(f"\n--- Running: {p_name} ---")
    
    for noise in noise_levels:
        if noise == 0.00:
            results[p_name]['stability'].append(1.0)
            results[p_name]['accuracy'].append(avg_baseline_acc)
            continue
            
        # 1. Generate Perturbed Dataset
        # Note: For Shuffle Features on IMDB-MULTI (OneHotDegree), 
        # features are generated on fly by transform. 
        # We apply perturbation AFTER transform (on the data object).
        noisy_data_list = []
        for d in dataset:
            # We must ensure d has x if we rely on OneHotDegree transform
            # The dataset[i] access applies transforms.
            d_perturbed = p_func(d, noise)
            noisy_data_list.append(d_perturbed)
            
        noisy_loader = DataLoader(noisy_data_list, batch_size=64, shuffle=False)
        
        # 2. Get Embeddings & Preds
        curr_embeddings = []
        curr_preds = []
        with torch.no_grad():
            for data in noisy_loader:
                data = data.to(device)
                _, probs, emb = model(data.x, data.edge_index, data.batch)
                curr_embeddings.append(emb)
                curr_preds.append(probs.argmax(dim=1))
        
        noisy_emb_tensor = torch.cat(curr_embeddings, dim=0)
        noisy_pred_tensor = torch.cat(curr_preds, dim=0)
        
        # 3. Compute Metrics
        # Cosine Similarity (Embedding Stability)
        cos_sim = F.cosine_similarity(base_emb_tensor, noisy_emb_tensor, dim=1).mean().item()
        
        # Accuracy
        acc = accuracy_score(base_labels_tensor.cpu(), noisy_pred_tensor.cpu())
        
        results[p_name]['stability'].append(cos_sim)
        results[p_name]['accuracy'].append(acc)
        
        print(f"Noise {int(noise*100)}% | Stability: {cos_sim:.4f} | Acc: {acc:.4f}")

# VISUALIZATION
plt.figure(figsize=(12, 5))
markers = {'Add Edges': 'o', 'Drop Edges': 's', 'Shuffle Features': '^', 'Mixed (Add+Drop)': 'D'}
colors  = {'Add Edges': 'b', 'Drop Edges': 'r', 'Shuffle Features': 'g', 'Mixed (Add+Drop)': 'm'}

# Plot 1: Embedding Stability
plt.subplot(1, 2, 1)
for name, data in results.items():
    plt.plot([n*100 for n in noise_levels], data['stability'], 
             marker=markers[name], color=colors[name], label=name, linewidth=2)
    
plt.title('Embedding Stability (Cosine Sim)')
plt.xlabel('Perturbation (%)')
plt.ylabel('Cosine Similarity')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.ylim(0, 1.05)

# Plot 2: Accuracy Degradation
plt.subplot(1, 2, 2)
for name, data in results.items():
    plt.plot([n*100 for n in noise_levels], data['accuracy'], 
             marker=markers[name], color=colors[name], label=name, linewidth=2)

plt.title('Accuracy Robustness')
plt.xlabel('Perturbation (%)')
plt.ylabel('Classification Accuracy')

plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()

plt.tight_layout()
plt.savefig('./plots/full_stability_analysis_{DATASET_NAME}_{EMBEDDING_SIZE}.png')
plt.show()

print("\nSaved full stability analysis to 'full_stability_analysis_{DATASET_NAME}_{EMBEDDING_SIZE}.png'")